In [2]:
import torch
import torch.nn as nn

### Simple Variation

In [4]:
class CausalSelfAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out=d_out
    self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self,x):
    b, n_tokens, d_in = x.shape
    keys=self.W_key(x)
    values=self.W_value(x)
    queries=self.W_query(x)
    attn_scores=queries@keys.transpose(1,2)
    attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weights=self.dropout(attn_weights)
    context_vec=attn_weights@values
    return context_vec

In [5]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
    super().__init__()
    self.heads = nn.ModuleList(
            [CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
    self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

  def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)

In [6]:
vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len

##### There ia an alternative implemnetation approach in which we will split query and keys

In [9]:
class MultiHeadtAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
    super().__init__()
    assert d_out%num_heads==0

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads
    self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.out_proj=nn.Linear(d_out,d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self,x):
    b,num_tokens,d_in=x.shape
    keys=self.W_key(x)
    values=self.W_value(x)
    queries=self.W_query(x)
    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values=values.view(b,num_tokens,self.num_heads,self.head_dim)
    queries=queries.view(b,num_tokens,self.num_heads,self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys=keys.transpose(1,2)
    values=values.transpose(1,2)
    queries=queries.transpose(1,2)

    attn_scores=queries@keys.transpose(2,3)
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
    attn_scores.masked_fill_(mask_bool, -torch.inf)

    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vec=attn_weights@values
    context_vec.transpose(1,2)
    context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
    context_vec = self.out_proj(context_vec)

    return context_vec

# 3. An alternative multi-head attention with combined weights


In [10]:
import torch.nn as nn
class MultiHeadAttentionCombinedQKV(nn.Module):
  def __init__(self,d_in,d_out,num_heads,context_length,dropout=0.0,qkv_bias=False):
    super().__init__()
    assert d_out/num_heads==0

    self.num_heads=num_heads
    self.d_out=d_out
    self.context_length=context_length
    self.head_dim=d_out//num_heads
    self.qkv=nn.Linear(d_in,3*d_out,bias=qkv_bias)
    self.out_proj=nn.Linear(d_out,d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

  def forward(self,x):
    batch_size,num_tokens,embed_dim=x.shape
    qkv=self.qkv(x)
    # (b,num_tokens,3*embed_dim)-->(b,num_tokens,3,num_heads,head_dim)
    qkv=qkv.view(batch_size,num_tokens,3,self.num_heads,self.head_dim)
    # (b,num_tokens,3,num_heads,head_dim)-->(3,b,num_heads,num_tokens,head_dim)
    qkv=qkv.permute(2,0,3,1,4)
    # (3,b,num_heads,num_tokens,head_dim)-->3 times(b,num_head,num_tokens,head_dim)
    queries,keys,values=qkv.unbind(0)
    # (b,num_heads,num_tokens,head_dim)-->(b,num_heads,num_tokens,num_tokens)
    attn_scores=queries@keys.transpose(-2,-1)
    attn_scores=attn_scores.masked_fill(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
    )
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weights=self.dropout(attn_weights)
    context_vec=attn_weights@values
    # (b,num_heads,num_tokens,head_dim)-->(b,num_tokens,num_heads,head_dim)
    context_vec = context_vec.transpose(1, 2)
    context_vec=context_vec.contiguous().view(batch_size,num_tokens,embed_dim)
    context_vec=self.proj(context_vec)
    return context_vec

# Using PyTorch's torch.nn.MultiheadAttention

In [11]:
class MHAPyTorchClass(nn.Module):
  def __init__(self,d_in,d_out,num_heads,context_length,dropout=0.0,qkv_bias=False,need_weights=True):
    super().__init__()
    self.context_length=context_length
    self.num_heads=num_heads
    self.multihead_attn=nn.MultiheadAttention(
        embed_dim=d_out,
        num_heads=num_heads,
        dropout=dropout,
        bias=qkv_bias,
        add_bias_kv=qkv_bias,
        batch_first=True,
    )
    self.need_weights=need_weights
    self.proj=nn.Linear(d_out,d_out)
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

  def forward(self,x):
    batch_size,num_tokens,emb_dim=x.shape
    if self.context_length >= num_tokens:
        attn_mask = self.mask[:num_tokens, :num_tokens]
    else:
        attn_mask = self.mask[:self.context_length, :self.context_length]
    attn_output, _ = self.multihead_attn(
        x, x, x, attn_mask=attn_mask, need_weights=self.need_weights
    )
    output=self.proj(attn_output)
    return output

# Implementing Multi Query Attention(MQA)  

In [1]:
import torch
import torch.nn as nn


class MultiQueryAttention(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False,
    ):
        super().__init__()

        assert d_out % num_heads == 0, (
            "d_out must be divisible by num_heads"
        )

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Multiple query heads
        self.W_query = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )

        # One shared key head and one shared value head
        self.W_key = nn.Linear(
            d_in, self.head_dim, bias=qkv_bias
        )
        self.W_value = nn.Linear(
            d_in, self.head_dim, bias=qkv_bias
        )

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length,
                    dtype=torch.bool,
                ),
                diagonal=1,
            ),
        )

    def forward(self, x):
        # x: [batch_size, num_tokens, d_in]
        b, num_tokens, _ = x.shape

        if num_tokens > self.mask.shape[0]:
            raise ValueError(
                "Input sequence exceeds context_length"
            )

        queries = self.W_query(x)
        # [b, num_tokens, d_out]

        keys = self.W_key(x)
        values = self.W_value(x)
        # Both: [b, num_tokens, head_dim]

        # Split queries into multiple heads
        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim
        )
        queries = queries.transpose(1, 2)
        # [b, num_heads, num_tokens, head_dim]

        # Singleton head dimension allows sharing by broadcasting
        keys = keys.unsqueeze(1)
        values = values.unsqueeze(1)
        # Both: [b, 1, num_tokens, head_dim]

        attn_scores = queries @ keys.transpose(-2, -1)
        # [b, num_heads, num_tokens, num_tokens]

        mask = self.mask[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(
            mask, float("-inf")
        )

        attn_weights = torch.softmax(
            attn_scores / self.head_dim**0.5,
            dim=-1,
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        # [b, num_heads, num_tokens, head_dim]

        # Combine head outputs
        context_vec = (
            context_vec.transpose(1, 2)
            .contiguous()
            .view(b, num_tokens, self.d_out)
        )
        # [b, num_tokens, d_out]

        return self.out_proj(context_vec)